# Measuring the bill

**Scenario:** a network operations team runs a fault triage assistant. It reads an alarm and asks
your code to send a field engineer. Finance asks one question. What does a fault cost to triage?

Nobody knows. The team picked its model off the price list. The bill disagrees.

Think of it as a taxi meter that runs while the driver thinks. The fare is not the distance you
travelled. It is the time the meter was running.

## Mechanics

Every response carries a usage block. The bill is built from these fields, and nothing else.

| Field | What it means |
|---|---|
| `usage.prompt_tokens` | Everything you sent, charged at the prompt rate |
| `usage.completion_tokens` | Everything the model produced, charged at the completion rate |
| `usage.completion_tokens_details.reasoning_tokens` | Part of that count you pay for and never see |
| `usage.total_tokens` | The two added up, which is not what you are billed on, because the rates differ |
| `build/provider-truth.json` | The two rates per model, written by `make probe` from the live API |

One row surprises people. Thinking is output. A model that reasons before it answers charges you
for the reasoning at the output rate, and none of that text reaches your code.

## The picture

![Rank models by cost per dispatched fault, not by price per token](images/measuring-the-bill.svg)

The price list and the meter are two different inputs. Only one of them knows what your prompt does
to a given model.

## The cost

```
usd  = prompt_tokens * prompt_rate + completion_tokens * completion_rate
unit = total usd / faults actually dispatched
```

The second line is the one that matters. A call that ends without dispatching anyone still costs
money, so dividing by calls flatters a model that often fails.

Both rates come from the probe, never from memory.

In [1]:
from vault import load_env, model_for, provider_truth

load_env()
FAST, THINKER = model_for("default"), model_for("reasoning")
rates = provider_truth()["models"]

for name in (FAST, THINKER):
    facts = rates[name]
    print(f"{name:30} prompt {float(facts['prompt_usd_per_token']):.2e}"
          f"   completion {float(facts['completion_usd_per_token']):.2e}")

google/gemini-2.5-flash-lite   prompt 1.00e-07   completion 4.00e-07
openai/gpt-5-nano              prompt 5.00e-08   completion 4.00e-07


## The failure

Read those two lines. One model is not dearer per token on either side of the bill, so the price
list picks it.

Here is the queue, and the one function the model may ask for. A tool call is the model asking your
code to run a named function, handed back as data.

In [2]:
DISPATCH = {"type": "function", "function": {
    "name": "dispatch_engineer",
    "description": "Send a field engineer. p1 total outage, p2 degraded, p3 single line.",
    "parameters": {"type": "object", "properties": {
        "fault_id": {"type": "string"},
        "site": {"type": "string"},
        "priority": {"type": "string", "enum": ["p1", "p2", "p3"]}},
        "required": ["fault_id", "site", "priority"],
        "additionalProperties": False}}}

FAULTS = [
    "Fault NF-3391 at site MAN-04: fibre cut, 12000 subscribers down.",
    "Fault NF-3392 at site LDS-11: packet loss at 4 percent, service degraded.",
    "Fault NF-3393 at site BRS-02: one business line dead since 09:00.",
    "Fault NF-3394 at site GLA-07: cell radio down, no traffic at all.",
]
SYSTEM = ("You triage telecom network faults. Dispatch an engineer with the tool. "
          "Do not ask questions.")

One call per fault. The meter reads the usage block and hands back three things: what it cost,
whether an engineer was dispatched, and how much of the output you never saw.

In [3]:
from vault import Usage, cost_of, get_client

client = get_client("03-token-economics/01-measuring-the-bill")


def triage(model, fault):
    """One fault, one call. Returns usage, whether it dispatched, and unseen tokens."""
    reply = client.chat.completions.create(
        model=model, max_tokens=800, tools=[DISPATCH],
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": fault}])
    details = reply.usage.completion_tokens_details
    unseen = getattr(details, "reasoning_tokens", 0) or 0
    return Usage.from_response(reply), bool(reply.choices[0].message.tool_calls), unseen

Now run the same queue through both models and compare the bill. The assertion states what the price
list promised.

In [4]:
runs = {model: [triage(model, fault) for fault in FAULTS] for model in (FAST, THINKER)}

for model, run in runs.items():
    out = sum(u.completion_tokens for u, _, _ in run)
    unseen = sum(n for _, _, n in run)
    spend = sum(cost_of(u) for u, _, _ in run)
    print(f"{model:30} out {out:5}  unseen {unseen:5}  ${spend:.6f}"
          f"  dispatched {sum(d for _, d, _ in run)}/{len(FAULTS)}")

bill = {model: sum(cost_of(u) for u, _, _ in run) for model, run in runs.items()}
assert bill[THINKER] <= bill[FAST], (
    f"the model with the lower price per token cost {bill[THINKER] / bill[FAST]:.1f} times more")

google/gemini-2.5-flash-lite   out   555  unseen   468  $0.000256  dispatched 4/4
openai/gpt-5-nano              out  1702  unseen  1408  $0.000705  dispatched 4/4


AssertionError: the model with the lower price per token cost 2.8 times more

## The diagnosis

The assertion fires, and the ratio is in the message.

**Thinking is billed as output.** The unseen column is reasoning. None of it reaches your code, and
all of it is charged at the completion rate.

**Both models reasoned, and one did far more of it.** Compare the two unseen counts. That gap is the
bill, and no price list contains it.

**Dividing by calls would have hidden it.** Finance asked for cost per fault dispatched. That number
only exists if your code counts the money and the work together.

Neither model has a defect. The cheaper rate was real. It was compared against the wrong unit.

## The fix

The fix is a unit. Dollars per call is an accident of how you batched the work. Dollars per fault
dispatched is the thing the business buys.

In [5]:
def unit_cost(run):
    """Dollars per fault actually dispatched, not dollars per call."""
    spend = sum(cost_of(u) for u, _, _ in run)
    dispatched = sum(d for _, d, _ in run)
    return spend / dispatched if dispatched else float("inf")

Next, the choice the team made before any of this ran. It reads the probe and ranks on rates alone,
which is what everyone does on day one.

In [6]:
def cheapest_by_sticker(models):
    """What the price list says, before anybody measures anything."""
    return min(models, key=lambda m: (float(rates[m]["completion_usd_per_token"]),
                                      float(rates[m]["prompt_usd_per_token"])))

Put the two verdicts side by side. Same queue, same day, two ways of deciding.

In [7]:
sticker = cheapest_by_sticker([FAST, THINKER])
measured = min(runs, key=lambda m: unit_cost(runs[m]))

for model in runs:
    print(f"{model:30} ${unit_cost(runs[model]):.6f} per dispatched fault")

print(f"\nprice list picks : {sticker}")
print(f"the meter picks  : {measured}")
print(f"gap              : {unit_cost(runs[sticker]) / unit_cost(runs[measured]):.1f}x per fault")

google/gemini-2.5-flash-lite   $0.000064 per dispatched fault
openai/gpt-5-nano              $0.000176 per dispatched fault

price list picks : openai/gpt-5-nano
the meter picks  : google/gemini-2.5-flash-lite
gap              : 2.8x per fault


Routing has to take the measured table, never the rates.

In [8]:
def pick_model(measured_unit_cost):
    """Rank by what a fault actually cost. The rates are not an input here."""
    return min(measured_unit_cost, key=measured_unit_cost.get)

## The gate

The regression to stop is somebody reintroducing the price list as a tie breaker. This test hands
`pick_model` a table where the two disagree, so it cannot pass by accident. It needs no API.

In [9]:
def test_the_price_list_does_not_pick_the_model():
    cheap_per_token, dear_per_token = "model-a", "model-b"
    measured_table = {cheap_per_token: 9.0, dear_per_token: 1.0}
    assert pick_model(measured_table) == dear_per_token


test_the_price_list_does_not_pick_the_model()
print("gate holds: the model with the lower rate does not win by default")

gate holds: the model with the lower rate does not win by default


Give `pick_model` the rates as a tie breaker and this test fails.

### Enterprise exploration

- The meter runs in the request path. Where do those rows go, and what does that cost at a thousand
  faults an hour?
- A model that reasons costs more per fault and may dispatch better. What would you have to measure
  to know whether the extra spend bought anything?
- The rates are read from a probe file. Who runs the probe, how often, and what happens to your
  budget the week a provider changes a price and nobody notices?
- Finance wants cost per fault by region and by customer tier. What has to be on every metered row
  for that to be answerable later?

### Key takeaways

- The bill is two rates and two counts. Read them off the response, not off a price list.
- Reasoning is output. You pay for it and you never see it.
- Cost per call flatters a model that fails. Cost per unit of work does not.
- Rank models on a measurement you took, on your prompt, with your tools.